# Decoder-Only --> GPT (Generative Pre-Trained Transformer)

* Optimized for Text Generation
"How are you? I am fine, thanks!" ----> Input: "How are you?" ---> Prediction: "I am fine, thanks!"
    * Can still be trained Few-Shot to learn new tasks without explicit labeling
    "What is the Translation for the portuguese word 'Aprendizado'? The translation is 'Learning' "
* GPT-2: Pre-LN (More stable) + Initialization + Optimized Scaling + Reddit Corpora + Optimized Tokenization ---> OpenAI's TikToken: Starts with 256 bytes to represent any string in bytes --> Adds common strings to vocabulary over time.
* GPT-3: "Language Models are Few-Shot Learners"
* (GPT-3.5): "Training language models to follow instructions with human feedback" (Reinforcement Learning with Human Feedback (RLHF))
* GPT-4: RLHF + Image Interpretation + Preliminary Reasoning `Let's break this down step-by-step` (https://openai.com/pt-BR/index/learning-to-reason-with-llms/)
* GPT-1o: https://openai.com/index/openai-o1-system-card/ - `The o1 large language model family is trained with reinforcement learning to perform complex reasoning. o1 thinks before it answers` ---> Possibly Text organized in such a way that make it more prolix, describing the entire reasoning method... OpenAI taking advantage of its capability of disposing of mechanical turks.

(PS: Avoid using OpenAI's blog in portuguese if you can...GPT translation can be...dubious...sometimes...)

In [1]:
import numpy as np
import torch
from torch import nn
import re
import math

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
TEXT_PATH = r"C:\Users\giova\OneDrive\Área de Trabalho\Faster than the Flame.txt"

In [4]:
text_data = []

with open(TEXT_PATH, 'r', encoding='utf-8') as f:

    for i in f:

        text_data.append(i)

f.close()
#text_data = [i.replace("\n", '') for i in text_data] # To avoid double \n in the following step
text_data = '\n'.join(text_data)
print(len(text_data))
print(text_data[0:200])

1415
Faster, faster, faster than the flame

Faster, faster, faster than the flame

Fists up in the air tonight

Leave the sane, unleash the wild

This is our time, this is our fate

Pyres will inflame the 


In [5]:
# Character Tokenization

tokens = ''.join(text_data)
tokens = ' '.join(tokens.split())
tokens = [i for i in tokens]

In [6]:
idx2char = ['\n', "<PAD>", "<EOS>"]

for character in tokens:

    if character not in idx2char:

        idx2char.append(character)

print(len(idx2char))
print(idx2char[:25])

43
['\n', '<PAD>', '<EOS>', 'F', 'a', 's', 't', 'e', 'r', ',', ' ', 'f', 'h', 'n', 'l', 'm', 'i', 'u', 'p', 'o', 'g', 'L', 'v', 'w', 'd']


In [7]:
class HeadAttention(nn.Module):

    def __init__(self, d_model, d_queries, d_values, masked=False):

        super(HeadAttention, self).__init__()

        self.d_queries = d_queries

        self.masked = masked

        self.create_queries = nn.Linear(d_model, d_queries, bias=False)
        self.create_values = nn.Linear(d_model, d_values, bias=False)
        self.create_keys = nn.Linear(d_model, d_values, bias=False)

        self.softmax = nn.Softmax(dim=-1)

    def forward(self, input, input_length=None): # Input length = index

        queries = self.create_queries(input) # (batch, sequences, d_queries)
        keys = self.create_keys(input) # (batch, sequences, d_keys)
        values = self.create_values(input) # (batch, sequences, d_values)

        similarity_matrix = torch.bmm(queries, keys.permute(0, 2, 1)) # (batch, sequences, sequences)

        similarity_matrix = similarity_matrix/(math.sqrt(self.d_queries))

        # Applying mask of -inf to ignore padded keys ---> Actually using -1e6 to avoid NaNs

        if self.masked:

            mask = torch.zeros_like(similarity_matrix, device=device)

            mask[:input_length+1] = 1 # +1 because "Shifted-Right"
            similarity_matrix = similarity_matrix.masked_fill(~mask.bool(), -1e6)

        attention_weights = self.softmax(similarity_matrix) # (batch, sequences, sequences)

        attention_output = torch.bmm(attention_weights, values) # (batch, sequences, d_values)

        return attention_output

In [8]:
class MultiHeadAttention(nn.Module):

    def __init__(self, n_heads, d_model, d_queries, d_values, masked=False):

        super(MultiHeadAttention, self).__init__()

        self.n_heads = n_heads
        self.attention_heads = nn.ModuleList([HeadAttention(d_model, d_queries, d_values, masked) for i in range(n_heads)])
        self.recompose_dimensions = nn.Linear(n_heads*d_values, d_model)

    def forward(self, input, input_length=None):

        output = []

        for head in range(len(self.attention_heads)):

            x = self.attention_heads[head](input, input_length)

            output.append(x)

        output = torch.cat(output, -1) # (batch, sequences, d_model*n_heads)

        output = self.recompose_dimensions(output) # (batch, sequences, d_model)

        return output

In [9]:
class PositionWiseFeedForward(nn.Module):

    def __init__(self, d_model, d_inner):

        super(PositionWiseFeedForward, self).__init__()

        self.neuron1 = nn.Linear(d_model, d_inner)
        self.Relu = nn.LeakyReLU()
        self.neuron2 = nn.Linear(d_inner, d_model)

    def forward(self, attention_output_cat):

        sequences = self.neuron1(attention_output_cat)
        sequences = self.Relu(sequences)
        output = self.neuron2(sequences)

        return output

In [10]:
class Decoder(nn.Module):

    def __init__(self, d_model, n_heads, d_queries, d_values, d_inner, dropout):

        super(Decoder, self).__init__()

        self.attentionA = MultiHeadAttention(n_heads, d_model, d_queries, d_values, masked=True)
        self.attentionB = MultiHeadAttention(n_heads, d_model, d_queries, d_values, masked=False)

        self.layer_norm = nn.LayerNorm(d_model)

        self.position_wise_neuron = PositionWiseFeedForward(d_model, d_inner)

        self.dropout = nn.Dropout(dropout)

    def forward(self, target_sequences, valid_positions):

        # valid_positions = mask tensor --> [1, 1, 1, 0, 0, 0]
        # 1 for valid (generated tokens), 0 for invalid (non-attended tokens)

        valid_positions = torch.sum(valid_positions).item()

        residual_block1 = target_sequences

        x = self.attentionA(target_sequences, valid_positions)

        x = self.dropout(x) + residual_block1
        x = self.layer_norm(x)

        residual_block2 = x

        x = self.position_wise_neuron(x)

        x = self.dropout(x) + residual_block2
        x = self.layer_norm(x)

        return x

In [11]:
class Transformer(nn.Module):

    def __init__(self, vocab_size, positional_encoding, d_model=512, n_heads=8, d_queries=64, d_values=64, d_inner=2056, n_layers=6, dropout=0.1):

        super(Transformer, self).__init__()

        self.n_layers = n_layers
        self.positional_encoding = positional_encoding
        self.d_model = d_model

        self.embedding = nn.Embedding(vocab_size, d_model)

        self.positional_encoding.requires_grad = False

        self.decoder = nn.ModuleList(
            Decoder(
                    d_model=d_model,
                    n_heads=n_heads,
                    d_queries=d_queries,
                    d_values=d_values,
                    d_inner=d_inner,
                    dropout=dropout) for i in range(self.n_layers)
        )

        self.output_neuron = nn.Linear(self.d_model, vocab_size)

        self.softmax = nn.LogSoftmax(-1)
                               
    def forward(self, input_sequences):

        input_sequences = self.embedding(input_sequences) * math.sqrt(self.d_model)

        input_sequences = input_sequences + self.positional_encoding.to(device)

        '''
        TEACHER ENFORCING:
        In the first Decoder, the target sequences are None (they aren't provided)
        For the following Decoders, we provide gradually the target sentences
        (pretending as if the previous decoders managed to predict them correctly)
        '''

        valid_positions = torch.zeros((input_sequences.size(1),)).long()

        for layer in range(self.n_layers):

            decoder_sequences = self.decoder[layer](input_sequences, valid_positions) # (batch, sequence, d_model)

            valid_positions[layer] = 1

        output = self.output_neuron(decoder_sequences) # (batch, sequence, vocab_size)

        output = self.softmax(output)

        return output

In [12]:
def get_positional_encoding(d_model, max_length=100):
    """
    Computes positional encoding as defined in the paper.
    :param d_model: size of vectors throughout the transformer model
    :param max_length: maximum sequence length up to which positional encodings must be calculated
    :return: positional encoding, a tensor of size (1, max_length, d_model)
    """
    positional_encoding = torch.zeros((1, max_length, d_model))  # (1, max_length, d_model)
    for i in range(max_length):
        for j in range(d_model):
            if j % 2 == 0:
                positional_encoding[:, i, j] = math.sin(i / math.pow(10000, j / d_model))
            else:
                positional_encoding[:, i, j] = math.cos(i / math.pow(10000, (j - 1) / d_model))

    return positional_encoding

In [13]:
positional_encoding = get_positional_encoding(d_model=64, max_length=24)

In [14]:
model = Transformer(
    vocab_size=len(idx2char),
    positional_encoding=positional_encoding,
    d_model=64,
    n_heads=8,
    d_queries=16,
    d_values=16,
    d_inner=256,
    n_layers=4,
    dropout=0.1
).to(device)

In [15]:
def get_batch(phrases, batch_size, sequence_length):

    sample_idx = torch.randint(len(phrases)-sequence_length, size=(batch_size,))

    x, y = [phrases[i:i+sequence_length-1] for i in sample_idx], [phrases[i+1:i+sequence_length] for i in sample_idx]

    minibatch, target = [], []

    # Adding <EOS> to Input sentence --> The model gets really confused without this token

    for sentence in x:
        k = []
        for character in sentence:
            k.append(idx2char.index(character))

        k = k + [idx2char.index("<EOS>")] + [idx2char.index("<PAD>")] * (sequence_length - len(k) - 1)
        minibatch.append(k)

    for sentence in y:
        k = []
        for character in sentence:
            k.append(idx2char.index(character))

        k = k + [idx2char.index("<PAD>")] * (sequence_length - len(k))
        target.append(k)

    minibatch, target = torch.tensor(minibatch), torch.tensor(target)

    return minibatch, target

In [16]:
x, y = get_batch(text_data, batch_size=64, sequence_length=24)

print(x.size())
print(y.size())

torch.Size([64, 24])
torch.Size([64, 24])


In [20]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.99), eps=1e-9, weight_decay=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=idx2char.index("<PAD>"))

iteration = 0
ITERATIONS = 10000

In [21]:
model.train()

while iteration < ITERATIONS:

    iteration += 1

    batch, target = get_batch(text_data, batch_size=64, sequence_length=24)

    batch, target = batch.to(device), target.to(device)

    model.zero_grad()

    output = model(batch)

    # Applying view seems such a lazy method...yet appears to work
    
    loss = criterion(output.view(-1, output.size(-1)), target.view(-1))

    loss.backward()

    optimizer.step()
        
    if iteration % 1000 == 0:

        print(f"Current Loss: {loss.item()}")
        print(f"{iteration}/{ITERATIONS}")

        output = output.max(-1)[1]

        decoded = [idx2char[output[0][i]] for i in range(output.size(-1))]
        phrase = [idx2char[target[0][i]] for i in range(target.size(-1))]

        decoded = ''.join(decoded)
        decoded = decoded.replace("<SOS>", "").replace("<EOS>", "").replace("<PAD>", "")
        phrase = ''.join(phrase)
        phrase = phrase.replace("<SOS>", "").replace("<EOS>", "").replace("<PAD>", "")
        print(f"Decoded Sentence: {decoded}")
        print(f"Target Sentence: {phrase}")

Current Loss: 0.3746573030948639
1000/10000
Decoded Sentence: e're going wild
WWhen w 
Target Sentence: e're going wild

When w
Current Loss: 0.35887062549591064
2000/10000
Decoded Sentence: dener


nflammmtum (flam
Target Sentence: dere)

Inflammatum (fla
Current Loss: 0.3104344606399536
3000/10000
Decoded Sentence: ght

Le ve she sane  un,
Target Sentence: ght

Leave the sane, un
Current Loss: 0.31918030977249146
4000/10000
Decoded Sentence: an the flame
FFiststup i
Target Sentence: an the flame

Fists up 
Current Loss: 0.2551434636116028
5000/10000
Decoded Sentence: s  th  wild
TThis is iui
Target Sentence: sh the wild

This is ou
Current Loss: 0.27978599071502686
6000/10000
Decoded Sentence:  faster, faster, fasterr
Target Sentence:  faster, faster, faster
Current Loss: 0.2705352306365967
7000/10000
Decoded Sentence: he flame
HHold the pastl
Target Sentence: he flame

Hold the past
Current Loss: 0.2773842215538025
8000/10000
Decoded Sentence: ame, flame, burngng winl
Target Sentenc

In [16]:
def inference(input_context, generation_window, sequence_length):
    '''
    Generates target sentence with time
    After target sentence reaches maximum length --> Shifts 1st token to input
    + discards 1st token in input
    '''

    input_context = [idx2char.index(character) for character in input_context]
    input_context += [idx2char.index("<EOS>")] + [idx2char.index("<PAD>")] * (sequence_length - len(input_context)-1)
    input_context = torch.tensor(input_context).unsqueeze(0).to(device)

    target = [idx2char.index("<PAD>")] * (input_context.size(-1))
    target = torch.tensor(target).unsqueeze(0).to(device)

    model.eval()

    generated_sentence = []

    with torch.no_grad():
        for i in range(generation_window):

            output = model(input_context)

            position_to_predict = min(i, sequence_length-1)
            output = output[0, position_to_predict, :]

            probs = torch.exp(output) # The output is in Log-scale (LogSoftmax)
            prediction = torch.multinomial(probs, num_samples=1).item()

            generated_sentence.append(prediction)

            if i < sequence_length:

                target[0, i] = prediction

            else:

                token_to_shift = target[0, 0].item()
                input_context = torch.roll(input_context, shifts=-1, dims=1)
                input_context[0, -1] = token_to_shift

                target = torch.roll(target, shifts=-1, dims=1)
                target[0, -1] = prediction

    generated_sentence = [idx2char[x] for x in generated_sentence]
    decoded = ''.join(generated_sentence).replace("<EOS>", "").replace("<PAD>", "")

    return generated_sentence, decoded

In [27]:
context = text_data[600:600+23]

print(context)

generated_sentence, decoded = inference(input_context=context, generation_window=40, sequence_length=24)

print(f"Target Sentence: {text_data[623:647]}")

print(generated_sentence)
print(f"Decoded Sentence: {decoded}")



Be prepared for sacri
Target Sentence: fice

This is our time, 
['\n', 'B', 'e', ' ', 'p', 'r', 'e', ' ', 'a', 'r', 'e', 'd', ' ', 's', 'o', 'r', 'i', 's', 'a', 'c', 'e', 'i', '\n', 'r', 'r', '\n', 'e', 'd', 's', 'a', 'i', '\n', 's', 'c', 'i', '\n', ' ', 's', 'a', 'r']
Decoded Sentence: 
Be pre ared sorisacei
rr
edsai
sci
 sar


In [17]:
TEXT_PATH = "D:/Python/Projects/Alice/shinamotaJP.txt"

In [18]:
text_data = []

with open(TEXT_PATH, 'r', encoding='utf-8') as f:

    for i in f:

        text_data.append(i)

f.close()
#text_data = [i.replace("\n", '') for i in text_data] # To avoid double \n in the following step
text_data = '\n'.join(text_data)
print(len(text_data))
print(text_data[0:200])

771
僕の命っつったって 誰の命っつったって

時時々 公平に 裁かれるもんなんでしょ

暗い空にやってきた 鬱を連れてやってきた

時々雨 総計に 頼り切りだ どうしよう

朽ちるまでの愛憎を 朽ちるまでの愛憎を

飲み込む君 簡単に 微笑む君 どうして

言葉を書く 曖昧に 言葉を書く 曖昧に

伝わりきらんないから 君だけをさ 信じて

捨ててきた夢をあつめて

ちょっと ちょっと 間違えたから



In [24]:
# Character Tokenization

tokens = ''.join(text_data)
tokens = ' '.join(tokens.split())
tokens = [i for i in tokens]

In [25]:
idx2char = ['\n', "<PAD>", "<EOS>"]

for character in tokens:

    if character not in idx2char:

        idx2char.append(character)

print(len(idx2char))
print(idx2char[:25])

121
['\n', '<PAD>', '<EOS>', '僕', 'の', '命', 'っ', 'つ', 'た', 'て', ' ', '誰', '時', '々', '公', '平', 'に', '裁', 'か', 'れ', 'る', 'も', 'ん', 'な', 'で']


In [26]:
positional_encoding = get_positional_encoding(d_model=64, max_length=24)

In [27]:
model = Transformer(
    vocab_size=len(idx2char),
    positional_encoding=positional_encoding,
    d_model=64,
    n_heads=8,
    d_queries=16,
    d_values=16,
    d_inner=256,
    n_layers=4,
    dropout=0.1
).to(device)

In [28]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.99), eps=1e-9, weight_decay=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=idx2char.index("<PAD>"))

iteration = 0
ITERATIONS = 10000

In [29]:
model.train()

while iteration < ITERATIONS:

    iteration += 1

    batch, target = get_batch(text_data, batch_size=64, sequence_length=24)

    batch, target = batch.to(device), target.to(device)

    model.zero_grad()

    output = model(batch)

    # Applying view seems such a lazy method...yet appears to work
    
    loss = criterion(output.view(-1, output.size(-1)), target.view(-1))

    loss.backward()

    optimizer.step()
        
    if iteration % 2000 == 0:

        print(f"Current Loss: {loss.item()}")
        print(f"{iteration}/{ITERATIONS}")

        output = output.max(-1)[1]

        decoded = [idx2char[output[0][i]] for i in range(output.size(-1))]
        phrase = [idx2char[target[0][i]] for i in range(target.size(-1))]

        decoded = ''.join(decoded)
        decoded = decoded.replace("<SOS>", "").replace("<EOS>", "").replace("<PAD>", "")
        phrase = ''.join(phrase)
        phrase = phrase.replace("<SOS>", "").replace("<EOS>", "").replace("<PAD>", "")
        print(f"Decoded Sentence: {decoded}")
        print(f"Target Sentence: {phrase}")

Current Loss: 0.24776677787303925
2000/10000
Decoded Sentence: つっつっつ 誰の命っつってって
時時時時 公平い
Target Sentence: つったって 誰の命っつったって

時時々 公平
Current Loss: 0.19010429084300995
4000/10000
Decoded Sentence: た 誰の命ってってって
時時時々 公平に 公かっ
Target Sentence: て 誰の命っつったって

時時々 公平に 裁か
Current Loss: 0.1711854785680771
6000/10000
Decoded Sentence: の？
ああ  夢を 夢を見てたはずが
怖
い  
Target Sentence: の？

ああ 夢を 夢を見てたはずが

怖い 
Current Loss: 0.1746482402086258
8000/10000
Decoded Sentence: ？
あああ 遠い夢を追いかけてさ

早い 早いい
Target Sentence: ？

ああ 遠い夢を追いかけてさ

早い 早い
Current Loss: 0.1545940637588501
10000/10000
Decoded Sentence:  追いつけないよ
捨捨てきれず残した思いが

て
Target Sentence:  追いつけないよ

捨てきれず残した思いが




In [31]:
context = text_data[600:600+23]

print(context)

generated_sentence, decoded = inference(input_context=context, generation_window=60, sequence_length=24)

print(f"Target Sentence: {text_data[623:647]}")

print(generated_sentence)
print(f"Decoded Sentence: {decoded}")

い 憎い 憎い 許されないの？

ああ 夢を 
Target Sentence: 夢を見てたはずが

怖い 怖い 怖い 怖い 怖い
[' ', '憎', 'い', ' ', '憎', 'い', ' ', '許', 'さ', 'れ', 'な', 'い', 'の', '？', '\n', '\n', 'あ', ' ', 'あ', '夢', 'を', ' ', '夢', 'を', 'な', '夢', 'い', 'の', '夢', 'い', '寄', '夢', 'さ', 'れ', 'な', 'い', 'の', 'い', '\n', 'あ', 'あ', ' ', '夢', ' ', 'を', ' ', '夢', 'を', ' ', 'い', 'を', 'の', '？', 'を', 'が', 'ら', 'を', 'れ', 'な', 'い']
Decoded Sentence:  憎い 憎い 許されないの？

あ あ夢を 夢をな夢いの夢い寄夢されないのい
ああ 夢 を 夢を いをの？をがらをれない
